# Day -3


1.   GroupBy: groupby + agg — survival rate by class, sex, age group
2.   Pivot tables, crosstabs — find patterns in categorical data


1.  Merge/join: inner, outer, left — join two DataFrames, handle key mismatches





•
•


In [70]:
import pandas as pd
##df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

In [71]:
#df = pd.read_csv('titanic_clean.csv')
import pandas as pd
import numpy as np

df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
df.fillna({'Age': df['Age'].median()}, inplace=True)
df.dropna(subset=['Embarked'], inplace=True)
df['Title'] = df['Name'].str.extract(', ([A-Za-z]+)\.')
df['AgeGroup'] = df['Age'].apply(lambda x: 'Child' if x<13 else 'Teen' if x<18 else 'Adult' if x<61 else 'Senior')
df['Sex_encoded'] = df['Sex'].map({'male':0, 'female':1})

<>:8: SyntaxWarning: invalid escape sequence '\.'
<>:8: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_584/2630965477.py:8: SyntaxWarning: invalid escape sequence '\.'
  df['Title'] = df['Name'].str.extract(', ([A-Za-z]+)\.')


In [72]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,AgeGroup,Sex_encoded
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Mr,Adult,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Mrs,Adult,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Miss,Adult,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Mrs,Adult,1
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Mr,Adult,0


### 1.   GroupBy: groupby + agg — survival rate by class, sex, age group

In [73]:
# 1.survival rate by class, sex, age group
# GroupBy - takes your entire DataFrame and splits it into separate
    #mini-DataFrames — one per unique value in the column you specify.
      #Then it applies a function to each mini-DataFrame independently. Then it combines the results back into one
df.groupby('Pclass')['Survived'].mean()

,Survived
Pclass,
1,0.626168
2,0.472826
3,0.242363


In [74]:
# survival rate grouped by both Pclass AND Sex at the same time
#df.groupby(df['Pclass','Sex'])['Survived'].mean()
df.groupby(['Pclass','Sex'])['Survived'].mean()

Pclass  Sex   
1       female    0.967391
        male      0.368852
2       female    0.921053
        male      0.157407
3       female    0.500000
        male      0.135447
Name: Survived, dtype: float64

In [75]:
# groupby-> age group
  # The Titanic Age column is continuous — actual ages like 22, 38, 7.
    #You can't groupby raw age. You need to bucket it first — child, adult, senior for example.
df['AgeGroup']=pd.cut(df['Age'],[0,18,59,99])
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,AgeGroup,Sex_encoded
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Mr,"(18, 59]",0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Mrs,"(18, 59]",1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Miss,"(18, 59]",1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Mrs,"(18, 59]",1
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Mr,"(18, 59]",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S,Rev,"(18, 59]",0
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S,Miss,"(18, 59]",1
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,28.0,1,2,W./C. 6607,23.4500,NaN,S,Miss,"(18, 59]",1
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C,Mr,"(18, 59]",0


In [76]:
#now groupby ageGroup
  # groupby-> age group
df.groupby('AgeGroup')['Survived'].mean()

/tmp/ipykernel_584/3781340142.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('AgeGroup')['Survived'].mean()


,Survived
AgeGroup,
"(0, 18]",0.503597
"(18, 59]",0.364138
"(59, 99]",0.240000


In [77]:
# agg()  --> Instead of one function, you can apply multiple at once.
        #So instead of just the mean survival rate, you might want — mean, count, and sum all in one go.
df.groupby('Pclass')['Survived'].agg(['mean', 'count', 'sum'])

,mean,count,sum
Pclass,,,
1,0.626168,214,134
2,0.472826,184,87
3,0.242363,491,119


 - pivot table is better when you're comparing two categorical variables against each other.

 - GroupBy is better when you want to chain more operations or aggregate multiple columns at once.

2.    Pivot tables, crosstabs — find patterns in categorical data

In [78]:
#Pivot tables
pd.pivot_table(df, values='Survived', index='Pclass', columns='Sex', aggfunc='mean')

Sex,female,male
Pclass,,
1,0.967391,0.368852
2,0.921053,0.157407
3,0.500000,0.135447


In [79]:
pd.pivot_table(df,values='Survived',index='Pclass',columns='Sex',aggfunc="sum")

Sex,female,male
Pclass,,
1,89,45
2,70,17
3,72,47


- pivot_tables ->work across columns of your dataframe




> Add blockquote


2.   List item



In [80]:
#crosstab
pd.crosstab(df['Pclass'], df['Sex'])

Sex,female,male
Pclass,,
1,92,122
2,76,108
3,144,347


3. Merge/join: inner, outer, left — join two DataFrames, handle key mismatches

In [95]:
df1 = pd.DataFrame({'PassengerId': [1, 2, 3, 4], 'Name': ['Alice', 'Bob', 'Charlie', 'Dave']})
df2 = pd.DataFrame({'PassengerId': [2, 2, 4, 5], 'Score': [85, 90, 78, 92]})

In [96]:
Merge_2 = df1.merge(df2,on = "PassengerId",how="left")
Merge_2

,PassengerId,Name,Score
0,1,Alice,NaN
1,2,Bob,85.0
2,2,Bob,90.0
3,3,Charlie,NaN
4,4,Dave,78.0


In [97]:
df1.merge(df2,on = "PassengerId",how="inner")

,PassengerId,Name,Score
0,2,Bob,85
1,2,Bob,90
2,4,Dave,78


In [85]:
df1.merge(df2,on = "PassengerId",how="outer")

,PassengerId,Name,Score
0,1,Alice,NaN
1,2,Bob,85.0
2,2,Bob,90.0
3,3,Charlie,NaN
4,4,Dave,78.0
5,5,NaN,92.0


In [86]:
#key mismatch

#Scenario 1 — different column names:
df3 = pd.DataFrame({'ID': [2, 3, 4, 5], 'Score': [85, 90, 78, 92]})

key_merged = df1.merge(df3,left_on="PassengerId",right_on="ID",how="outer")
key_merged

,PassengerId,Name,ID,Score
0,1.0,Alice,NaN,NaN
1,2.0,Bob,2.0,85.0
2,3.0,Charlie,3.0,90.0
3,4.0,Dave,4.0,78.0
4,NaN,NaN,5.0,92.0


In [87]:
#key mismatch
# Scenario 2 — duplicate keys
df1.merge(df2,on = "PassengerId",how="outer")

,PassengerId,Name,Score
0,1,Alice,NaN
1,2,Bob,85.0
2,2,Bob,90.0
3,3,Charlie,NaN
4,4,Dave,78.0
5,5,NaN,92.0


# Day 3 done



In [88]:
"""
Issues as two similar columns with same values.
"""

#Scenario 1 — you got both columns kept (PassengerId AND ID).
#In real code you'd drop the redundant one after merging.
key_merged.drop("ID",axis=1)

,PassengerId,Name,Score
0,1.0,Alice,NaN
1,2.0,Bob,85.0
2,3.0,Charlie,90.0
3,4.0,Dave,78.0
4,NaN,NaN,92.0


- Scenario 2 — you predicted it exactly. Duplicate key → row gets multiplied.
1. This is called a one-to-many join and it's a common source of bugs in data pipelines. Always check for duplicates before merging.

In [89]:
Merge_2

,PassengerId,Name,Score
0,1,Alice,NaN
1,2,Bob,85.0
2,2,Bob,90.0
3,3,Charlie,NaN
4,4,Dave,78.0


In [98]:
print(df2['PassengerId'].duplicated().sum())

1


In [99]:
# see which rows are duplicated
df2[df2['PassengerId'].duplicated(keep=False)]

,PassengerId,Score
0,2,85
1,2,90


In [90]:
"""

"""
# Merge_2
Merge_2.drop_duplicates(subset=["PassengerId"],inplace=True)
Merge_2

,PassengerId,Name,Score
0,1,Alice,NaN
1,2,Bob,85.0
3,3,Charlie,NaN
4,4,Dave,78.0
